# Computer Engineering Department ML Project SOP
## Week 2: Data Cleaning and Preprocessing
**Objective:** Handle missing values, identify and handle outliers, encode categorical variables, normalize/scale numerical features, and perform comprehensive Exploratory Data Analysis (EDA).

---


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, LabelEncoder

sns.set_theme(style="whitegrid", palette="muted")
df = pd.read_csv('../data/Loan_default.csv')
if 'LoanID' in df.columns:
    df.drop(columns=['LoanID'], inplace=True)
print("Dataset loaded successfully. Shape:", df.shape)


### 1. Missing Values Verification & Pipeline
We verify whether null, NaN, or sentinel empty values exist in any column.


In [ ]:
missing_summary = df.isnull().sum()
print("Missing values per feature:")
print(missing_summary[missing_summary > 0] if missing_summary.sum() > 0 else "No missing values found (100% complete dataset).")


### 2. Outlier Detection and Handling
We employ the **Interquartile Range (IQR)** method to detect extreme outliers in continuous financial variables:
$$\text{IQR} = Q_3 - Q_1, \quad \text{Lower Bound} = Q_1 - 1.5 \cdot \text{IQR}, \quad \text{Upper Bound} = Q_3 + 1.5 \cdot \text{IQR}$$


In [ ]:
numerical_cols = ['Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed', 'InterestRate', 'DTIRatio']

plt.figure(figsize=(14, 6))
for i, col in enumerate(numerical_cols[:4], 1):
    plt.subplot(1, 4, i)
    sns.boxplot(y=df[col], color='#4575b4')
    plt.title(f"{col} Distribution")
plt.tight_layout()
plt.show()

# Detect and display outlier counts
for col in numerical_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    print(f"{col:15s}: {len(outliers):6d} outliers detected ({len(outliers)/len(df)*100:.2f}%)")


### 3. Categorical Variable Encoding
- **Binary Features:** Label encode (`No` -> 0, `Yes` -> 1) for `HasMortgage`, `HasDependents`, `HasCoSigner`.
- **Nominal Features:** One-Hot Encode (`Education`, `EmploymentType`, `MaritalStatus`, `LoanPurpose`) into 0/1 dummy indicator columns.


In [ ]:
# Binary encoding
binary_cols = ['HasMortgage', 'HasDependents', 'HasCoSigner']
df_encoded = df.copy()
for col in binary_cols:
    df_encoded[col] = df_encoded[col].apply(lambda x: 1 if str(x).strip().lower() in ['yes', '1', 'true'] else 0)

# One-hot encoding
nominal_cols = ['Education', 'EmploymentType', 'MaritalStatus', 'LoanPurpose']
df_encoded = pd.get_dummies(df_encoded, columns=nominal_cols, drop_first=False)
print("Processed feature count after one-hot encoding:", df_encoded.shape[1] - 1)
df_encoded.head()


### 4. Numerical Feature Normalization / Scaling
Comparing `StandardScaler` (z-score standardization) vs `MinMaxScaler` on continuous distributions.


In [ ]:
scaler_std = StandardScaler()
scaler_minmax = MinMaxScaler()

scaled_std = scaler_std.fit_transform(df[numerical_cols])
scaled_minmax = scaler_minmax.fit_transform(df[numerical_cols])

print("StandardScaler sample output (Mean ~ 0, Std ~ 1):")
print(scaled_std[:2, :4])


### 5. Exploratory Data Analysis (EDA) Visualizations
Analyzing key bivariate relationships with the loan default outcome.


In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.boxplot(x='Default', y='InterestRate', data=df, palette=['#4575b4', '#d73027'])
plt.title("Interest Rate by Default Status")
plt.xticks([0, 1], ["Repaid (0)", "Default (1)"])

plt.subplot(1, 2, 2)
sns.boxplot(x='Default', y='Income', data=df, palette=['#4575b4', '#d73027'])
plt.title("Income by Default Status")
plt.xticks([0, 1], ["Repaid (0)", "Default (1)"])

plt.tight_layout()
plt.show()


In [ ]:
# Correlation Heatmap among numerical features
plt.figure(figsize=(10, 8))
corr = df[numerical_cols + ['Default']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', vmin=-0.2, vmax=0.2, cbar=True)
plt.title("Correlation Matrix: Numerical Features vs Default")
plt.show()
